# Explainable Boosting Model

InterpretML の EBM で入場者数モデルを解釈する。


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while (
    PROJECT_ROOT != PROJECT_ROOT.parent
    and not (PROJECT_ROOT / "requirements.txt").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"


In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv(PROCESSED_DATA_DIR / "to_ML.csv")


In [ ]:
df.columns


In [ ]:
# 必要な変数セット
target_cols = [
    "コロナ禍ダミー",
    "国立フラグ",
    "休日フラグ",
    "rolling_mean_3",
    "rain_flag",
    "temp_zone",
    "month",
]

# 全データのX
X_all = df[target_cols].dropna()

# 全データのy
y_all = df.loc[X_all.index, "入場者数"]

# 学習データとテストデータの分割
train_mask = df.loc[X_all.index, "シーズン"].astype(str).str.contains("2024") == False
test_mask = df.loc[X_all.index, "シーズン"].astype(str).str.contains("2024") == True

X_train = X_all[train_mask]
y_train = y_all[train_mask]

X_test = X_all[test_mask]
y_test = y_all[test_mask]

print(f"学習データ数: {len(X_train)}")
print(f"テストデータ数: {len(X_test)}")


In [ ]:
X_train


In [ ]:
%pip install interpret


In [ ]:
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show

# データ準備（X_train, y_train = 観客数）
ebm = ExplainableBoostingRegressor(random_state=42)
ebm.fit(X_train, y_train)

y_pred = ebm.predict(X_test)


In [ ]:
show(ebm.explain_global(name="EBM Global"))


In [ ]:
show(ebm.explain_local(X_test[:5], y_test[:5], name="EBM Local"))


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:   ", r2_score(y_test, y_pred))


In [ ]:
import matplotlib.pyplot as plt

# フォント設定
plt.rcParams["font.family"] = "Hiragino Sans"

# ---------------------------------------------------------
# 5. 答え合わせ（可視化）
# ---------------------------------------------------------
# 精度スコアの計算
r2_test = r2_score(y_test, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))

print("========================================")
print(f"★ 2024年予測の精度 (R2): {r2_test:.3f}")
print(f"★ 誤差の大きさ (RMSE)  : {rmse_test:,.0f} 人")
print("========================================")

# 散布図で確認
plt.figure(figsize=(6, 6))

# 誤差±20%のエリアを描画
max_val = max(y_test.max(), y_pred.max())
x_line = np.linspace(0, max_val, 100)
plt.fill_between(
    x_line, x_line * 0.8, x_line * 1.2, alpha=0.2, color="green", label="±20% 誤差範囲"
)

plt.scatter(y_test, y_pred, color="blue", alpha=0.6, label="2024 Matches")

# 真ん中の線（ピッタリ賞のライン）
plt.plot([0, max_val], [0, max_val], color="red", linestyle="--", label="Perfect Fit")

plt.title("実際 vs 予測 (2024年)")
plt.xlabel("実際の入場者数")
plt.ylabel("予測の入場者数")
plt.legend()
plt.grid(True)
plt.show()

# 具体的にどの試合をどう外したか見る
results_df = pd.DataFrame(
    {
        "対戦相手": df.loc[X_test.index, "アウェイ"],
        "スタジアム": df.loc[X_test.index, "スタジアム"],
        "実測値": y_test,
        "予測値": y_pred,
        "誤差": y_pred - y_test,
    }
)
display(results_df)
